# Attention Mechanism for Time-Series: A Transparent Toy Example

**Course alignment:** Time-Series Forecasting and Attention Mechanism  
**Type:** Original supplementary practice

The course slides describe Query-Key scores, softmax weights and a weighted sum of Values. This notebook builds a small numerical example from scratch to show how an attention layer can assign different weights to historical time steps. It is a conceptual illustration, not a trained forecasting model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# A short synthetic load sequence. Each value is a feature vector
# representing one time step (e.g. demand, temperature, calendar signal).
history=np.array([
    [40.,25.,0.], [42.,26.,0.], [39.,24.,0.],
    [55.,29.,1.], [58.,30.,1.], [43.,25.,0.],
    [41.,24.,0.], [57.,30.,1.]
])
feature_names=["load","temperature","peak_indicator"]
pd.DataFrame(history,columns=feature_names,index=[f"t-{len(history)-i}" for i in range(len(history))])

## 1. Scaled dot-product attention

Given a query vector Q and keys K, attention scores are QKᵀ/√d. Softmax converts scores to weights that sum to one. The weights are applied to value vectors V to produce a context vector.

In [ ]:
# Normalise the feature dimensions before comparing dot products.
# This prevents the raw load scale from dominating temperature and peak signals.
feature_mean=history.mean(axis=0)
feature_std=history.std(axis=0)
normalised=(history-feature_mean)/feature_std

d=normalised.shape[1]
# A query representing a high-load, warm, peak-period context in standardised space.
query=np.array([.8,.5,1.0])
keys=normalised.copy()
values=normalised.copy()

scores=(keys @ query)/np.sqrt(d)
weights=softmax(scores)
context_standardised=weights @ values
context=context_standardised*feature_std+feature_mean

print("Attention weights:",np.round(weights,3))
print("Sum of weights:",weights.sum())
print("Weighted historical context:",np.round(context,2))


## 2. Inspect which past time steps receive attention

In [ ]:
labels=[f"t-{len(history)-i}" for i in range(len(history))]
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(labels,weights)
ax.set(title="Attention weight by historical time step",xlabel="Historical time step",ylabel="Attention weight")
plt.tight_layout()
plt.show()

for label,weight in zip(labels,weights):
    print(label, f"{weight:.3f}")

## 3. What attention does and does not mean

Attention provides a learned weighting mechanism over representations. In a trained model, query, key and value projections are learned from data. The weights are conditional on the query and model state; they should not automatically be interpreted as causal effects or definitive feature importance.

A practical attention-based forecaster would also need:
- chronological training and validation;
- a suitable sequence window and forecast horizon;
- positional/time encodings or explicit time features;
- regularisation and early stopping;
- comparison against seasonal-naive, smoothing, Prophet and SARIMAX baselines;
- uncertainty analysis and monitoring after deployment.

## Exercise

Change the query to emphasise lower temperature or a non-peak day, observe how the attention distribution changes, and explain why this is only a hand-designed demonstration rather than a model trained to forecast.